# Notebook 4: 03_eda_analysis
Six EDA charts for class, noise, FP source, scene FP rate, time FP rate, and score distribution.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path


In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
FIG_DIR = PROJECT_ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_PATH = DATA_DIR / 'audio_clip_event_cleaned.csv'
if not AUDIO_PATH.exists():
    AUDIO_PATH = DATA_DIR / 'audio_clip_event.csv'

df = pd.read_csv(AUDIO_PATH, parse_dates=['event_time'])
print('Using:', AUDIO_PATH)
print('Shape:', df.shape)


In [ ]:
# Figure 1: sample distribution
cnt = df['binary_label'].value_counts().sort_index()
labels = ['Negative (0)', 'Positive (1)']
values = [int(cnt.get(0,0)), int(cnt.get(1,0))]
plt.figure(figsize=(7,4))
plt.bar(labels, values, color=['#6b7280','#ef4444'])
for i,v in enumerate(values):
    plt.text(i, v+50, str(v), ha='center')
plt.title('Sample Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIG_DIR/'sample_distribution.png', dpi=140)
plt.show()


In [ ]:
# Figure 2: noise class distribution
neg = df[df['binary_label']==0]
noise = neg['true_class'].value_counts()
plt.figure(figsize=(9,4))
noise.plot(kind='bar', color='#60a5fa')
plt.title('Noise Class Distribution')
plt.ylabel('Count')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR/'noise_class_distribution.png', dpi=140)
plt.show()


In [ ]:
# Figure 3: FP root cause
fp = df[df['error_type']=='FP']['true_class'].value_counts()
plt.figure(figsize=(9,4))
fp.plot(kind='bar', color='#f59e0b')
plt.title('FP Root Cause Distribution')
plt.ylabel('FP Count')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR/'fp_root_cause.png', dpi=140)
plt.show()


In [ ]:
# Figure 4: FP rate by scene
scene = df.groupby('scene_type').apply(lambda x: pd.Series({'total':len(x),'fp':int((x.error_type=='FP').sum())})).reset_index()
scene['fp_rate'] = scene['fp'] / scene['total']
scene = scene.sort_values('fp_rate', ascending=False)
plt.figure(figsize=(8,4))
plt.bar(scene['scene_type'], scene['fp_rate'], color='#34d399')
plt.title('FP Rate by Scene')
plt.ylabel('FP Rate')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR/'scene_fp_rate.png', dpi=140)
plt.show()
scene


In [ ]:
# Figure 5: FP rate by time bucket
bucket = df.groupby('time_bucket').apply(lambda x: pd.Series({'total':len(x),'fp':int((x.error_type=='FP').sum())})).reset_index()
bucket['fp_rate'] = bucket['fp'] / bucket['total']
order = ['daytime','evening_peak','night','early_morning']
bucket['time_bucket'] = pd.Categorical(bucket['time_bucket'], categories=order, ordered=True)
bucket = bucket.sort_values('time_bucket')
plt.figure(figsize=(8,4))
plt.bar(bucket['time_bucket'].astype(str), bucket['fp_rate'], color='#a78bfa')
plt.title('FP Rate by Time Bucket')
plt.ylabel('FP Rate')
plt.tight_layout()
plt.savefig(FIG_DIR/'time_bucket_fp_rate.png', dpi=140)
plt.show()
bucket


In [ ]:
# Figure 6: score distribution by TP/FP/FN/TN
plt.figure(figsize=(9,5))
for err,color in [('TP','#22c55e'),('FP','#ef4444'),('FN','#f97316'),('TN','#3b82f6')]:
    vals = df.loc[df['error_type']==err, 'model_score']
    plt.hist(vals, bins=35, alpha=0.45, density=True, label=err, color=color)
plt.axvline(0.80, color='black', linestyle='--', linewidth=1, label='threshold=0.8')
plt.title('Score Distribution by Error Type')
plt.xlabel('Model Score')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR/'score_distribution.png', dpi=140)
plt.show()
